# Kolam Lelehan Arktik dengan Model Ising

**ID proyek:** `O005-LEGA-V101-PRJ12`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Bagaimana kopling tetangga, pemaksaan eksternal, dan jadwal pendinginan mengubah pola biner es–kolam dalam model Ising pedagogis?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082212
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Kisi periodik memetakan spin +1 ke kolam dan −1 ke es; energi tetangga dan medan eksternal mengatur pembalikan; pembaruan Metropolis berurutan acak.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
L, coupling, field = 30, 1.0, 0.10
spins = rng.choice(np.array([-1, 1], dtype=int), size=(L, L))
initial_spins = spins.copy()

def ising_energy(state):
    neighbor_sum = np.roll(state, 1, axis=0) + np.roll(state, 1, axis=1)
    return float(-coupling * np.sum(state * neighbor_sum) - field * np.sum(state))

initial_energy = ising_energy(spins)
energy_history = [initial_energy]
temperature_schedule = np.linspace(2.8, 0.45, 42)
for temperature in temperature_schedule:
    for _ in range(L * L):
        i, j = rng.integers(0, L, size=2)
        neighbor = spins[(i - 1) % L, j] + spins[(i + 1) % L, j] + spins[i, (j - 1) % L] + spins[i, (j + 1) % L]
        delta = 2.0 * spins[i, j] * (coupling * neighbor + field)
        if delta <= 0.0 or rng.random() < np.exp(-delta / temperature):
            spins[i, j] *= -1
    energy_history.append(ising_energy(spins))

# Relaksasi rakus membuat pemeriksaan energi akhir deterministik dan transparan.
for _ in range(8):
    for i in range(L):
        for j in range(L):
            neighbor = spins[(i - 1) % L, j] + spins[(i + 1) % L, j] + spins[i, (j - 1) % L] + spins[i, (j + 1) % L]
            delta = 2.0 * spins[i, j] * (coupling * neighbor + field)
            if delta < 0.0:
                spins[i, j] *= -1
    energy_history.append(ising_energy(spins))
final_energy = ising_energy(spins)
melt_fraction = float(np.mean(spins == 1))


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
assert set(np.unique(spins)).issubset({-1, 1})
assert final_energy <= initial_energy + 1e-12
assert 0.0 <= melt_fraction <= 1.0
assert np.isfinite(energy_history).all()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5))
axes[0].imshow(initial_spins, cmap="Blues", vmin=-1, vmax=1)
axes[0].set(title="kisi awal")
axes[1].imshow(spins, cmap="Blues", vmin=-1, vmax=1)
axes[1].set(title=f"kisi akhir; kolam={melt_fraction:.2f}")
axes[2].plot(energy_history)
axes[2].set(xlabel="sapuan", ylabel="energi", title="relaksasi energi")
for ax in axes[:2]:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Spin bukan hidrologi fisik; tidak ada konservasi air, ketebalan es, geometri nyata, radiasi, aliran, atau kalibrasi observasional.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
